## Theory: LSTM-SNP Architecture

The LSTM-SNP model uses a custom recurrent cell inspired by Spiking Neural P (SNP) systems.

### Gate Equations

$$r(t) = \rho(W_r x(t) + U_r u(t-1) + b_r)$$ — Reset gate

$$c(t) = \rho(W_c x(t) + U_c u(t-1) + b_c)$$ — Consumption gate

$$o(t) = \rho(W_o x(t) + U_o u(t-1) + b_o)$$ — Output gate

$$a(t) = f(W_a x(t) + U_a u(t-1) + b_a)$$ — Generated spikes

### State Update

$$u(t) = r(t) \cdot u(t-1) - c(t) \cdot a(t)$$

$$h(t) = o(t) \cdot a(t)$$

where $\rho$ = hard sigmoid, $f$ = tanh

# Baseline LSTM-SNP — Dow Jones Industrial Index
## + Gaussian Input-Noise Robustness (0.5% / 5% / 10% / 15%)

x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = eta * std(x)

Noise is injected into input features only (never targets), so the resulting
metrics reflect model sensitivity to input perturbation, not label corruption.

This version includes: a corrected NMSE denominator, validation-based early
stopping, gradient clipping, and noise-sweep averaging over multiple ε(t)
draws with a paired per-model degradation test.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from math import sqrt
from scipy import stats
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.11.0+cu128
NumPy version: 2.5.1
Using device: cuda


## Model Architecture & Implementation

### LSTM-SNP Cell

In [2]:
#with noise


class LSTMSNPCell(nn.Module):
    """
    LSTM-SNP Cell: gates r, c, o (hard sigmoid) and generated spikes a (tanh).
    u(t) = r(t)*u(t-1) - c(t)*a(t)
    h(t) = o(t)*a(t)
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.kernel = nn.Linear(input_size, hidden_size * 4, bias=False)
        self.recurrent_kernel = nn.Linear(hidden_size, hidden_size * 4, bias=False)
        self.bias = nn.Parameter(torch.zeros(hidden_size * 4))

        nn.init.xavier_uniform_(self.kernel.weight)
        nn.init.orthogonal_(self.recurrent_kernel.weight)

    def hard_sigmoid(self, x):
        return torch.clamp(0.2 * x + 0.5, 0.0, 1.0)

    def forward(self, x, u_tm1):
        z = self.kernel(x) + self.recurrent_kernel(u_tm1) + self.bias
        z0, z1, z2, z3 = z.chunk(4, dim=-1)

        r = self.hard_sigmoid(z0)   # reset
        c = self.hard_sigmoid(z1)   # consumption
        o = self.hard_sigmoid(z2)   # output/generation
        a = torch.tanh(z3)          # generated spikes

        u = r * u_tm1 - c * a
        h = o * a
        return h, u

### Build Model

In [3]:
#with noise
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = LSTMSNPCell(input_size, hidden_size)
        self.out = nn.Linear(hidden_size, 1)
        self.u = None

    def reset_states(self, batch_size, device):
        self.u = torch.zeros(batch_size, self.hidden_size, device=device)

    def forward(self, x):
        # x is (batch, 1, input_size)
        if self.u is None or self.u.device != x.device:
            self.reset_states(x.size(0), x.device)
        h, self.u = self.cell(x[:, 0, :], self.u)
        return self.out(h)


def build_model(input_dim, units):
    return RNNModel(input_dim, units)

In [4]:
#with noise
model = build_model(input_dim=1, units=8).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

RNNModel(
  (cell): LSTMSNPCell(
    (kernel): Linear(in_features=1, out_features=32, bias=False)
    (recurrent_kernel): Linear(in_features=8, out_features=32, bias=False)
  )
  (out): Linear(in_features=8, out_features=1, bias=True)
)

Total params: 329
Trainable params: 329


## Data Pipeline — Dow Jones Industrial Index

In [5]:
DATA_PATH = r'C:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\dataset\sp500.csv'

series = pd.read_csv(DATA_PATH, header=0, parse_dates=[0], index_col=0)
raw_values = series.values.flatten()

In [6]:
# ============================================================
# 2. First-Order Differencing
# ============================================================

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        value = dataset[i] - dataset[i - interval]
        diff.append(value)
    return np.array(diff)

diff_values = difference(raw_values, 1)

In [7]:
# ============================================================
# 3. Convert to Supervised Learning Format (lag=1)
# ============================================================

def timeseries_to_supervised(data, lag=1):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

supervised = timeseries_to_supervised(diff_values, 1)
print(f"Supervised data shape: {supervised.shape}")

Supervised data shape: (250, 2)


In [8]:
#with noise Train-Validation-Test Split (Last 60 points as Test)
n = len(supervised)
test_size = 60
train_val = supervised[:n - test_size]
test = supervised[n - test_size:]

val_size = int(len(train_val) * 0.1)
train = train_val[:-val_size]
val = train_val[-val_size:]

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")

scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train)  # fit only on train to avoid leakage
train_scaled = scaler.transform(train)
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

Train: (171, 2), Val: (19, 2), Test: (60, 2)


In [9]:
# ============================================================
# 6. Reshape for RNN Input
# ============================================================

X_train, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))

X_test, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (171, 1, 1), y_train shape: (171,)


In [10]:
#Gaussian noise injection (Eq. 29–30)
ref_std = X_train.std()
print(f"Reference std(x) used for noise scaling (train inputs): {ref_std:.6f}")

NOISE_LEVELS = [0.005, 0.05, 0.10, 0.15]  # 0.5%, 5%, 10%, 15%

def add_gaussian_noise(X, noise_level, ref_std, seed=None):
    """
    x_noisy(t) = x(t) + eps(t),  eps(t) ~ N(0, sigma_eps^2),  sigma_eps = noise_level * std(x)
    Applied to input features X only — never to targets/labels.
    """
    rng = np.random.default_rng(seed)
    sigma_eps = noise_level * ref_std
    eps = rng.normal(loc=0.0, scale=sigma_eps, size=X.shape)
    return X + eps

Reference std(x) used for noise scaling (train inputs): 0.235521


In [11]:
#Evaluation function

def evaluate_on_test(model, X_test_eval, train, train_scaled, raw_values, scaler, device):
    """
    Warms up the model's recurrent state on clean training data, then runs
    single-step-ahead prediction on X_test_eval (clean or Gaussian-noise-corrupted
    input features). Test targets are always the clean, ground-truth values.
    """
    model.eval()
    model.reset_states(1, device)

    with torch.no_grad():
        for i in range(len(train_scaled)):
            X_raw = train_scaled[i, 0:-1]
            X_input = torch.tensor(X_raw, dtype=torch.float32).view(1, 1, len(X_raw)).to(device)
            model(X_input)

        predictions = []
        for i in range(len(X_test_eval)):
            X = X_test_eval[i]
            X_input = torch.tensor(X, dtype=torch.float32).view(1, 1, len(X)).to(device)
            yhat = model(X_input).item()

            new_row = [x for x in X] + [yhat]
            array = np.array(new_row).reshape(1, len(new_row))
            inverted = scaler.inverse_transform(array)[0, -1]
            inverted = inverted + raw_values[len(train) + i]
            predictions.append(inverted)

    actual = raw_values[-len(X_test_eval):]
    mse = mean_squared_error(actual, predictions)
    rmse = sqrt(mse)
    meanV = np.mean(actual)
    dominator = np.linalg.norm(np.array(actual) - meanV, 2)   # FIX: was predictions - meanV
    nmse = mse / np.power(dominator, 2)
    return predictions, rmse, mse, nmse

#Training loop

In [12]:
import os

#Training loop (60 runs, early stopping + gradient clipping) -- with checkpointing
#
# Checkpointing: after every run, the trained (best-val) model's state_dict +
# that run's metrics are written to disk under CHECKPOINT_DIR. If this cell is
# re-run (kernel restart, crash, "Restart & Run All", etc.), it picks up from
# the last completed run instead of retraining from scratch.

CHECKPOINT_DIR = "checkpoints_sp500"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _ckpt_path(run):
    return os.path.join(CHECKPOINT_DIR, f"run_{run}.pt")

def _summary_path():
    return os.path.join(CHECKPOINT_DIR, "summary.pt")

all_rmse = []
all_mse = []
all_nmse = []
all_predictions = []
all_losses = []
all_models = []  # keep every trained model for later noise-robustness eval

print(f"\n--- [PyTorch] RUNNING ON {device} ---\n")
print(f"Checkpoints will be read/written under: {os.path.abspath(CHECKPOINT_DIR)}\n")

N_RUNS = 60
val_X = val_scaled[:, 0:-1]
PATIENCE = 10

# If a fully-completed summary already exists (e.g. all 60 runs finished in a
# previous session), just reload it and skip training entirely.
summary_path = _summary_path()
if os.path.exists(summary_path):
    print("Found completed checkpoint summary -> loading, skipping training.")
    saved = torch.load(summary_path, map_location=device, weights_only=False)

    all_models = []
    for sd in saved['model_state_dicts']:
        m = build_model(input_dim=1, units=8).to(device)
        m.load_state_dict(sd)
        m.eval()
        all_models.append(m)

    all_rmse = saved['rmse']
    all_mse = saved['mse']
    all_nmse = saved['nmse']
    all_predictions = saved['predictions']
    all_losses = saved['losses']
    print(f"Loaded {len(all_models)} models from checkpoint.")

else:
    for run in range(N_RUNS):

        # Resume mid-run-set: if this specific run was already checkpointed
        # (e.g. the kernel died partway through), reload it instead of
        # retraining.
        run_ckpt_path = _ckpt_path(run)
        if os.path.exists(run_ckpt_path):
            print(f'\n===== RUN {run + 1}/{N_RUNS} (resumed from checkpoint) =====')
            ckpt = torch.load(run_ckpt_path, map_location=device, weights_only=False)

            model = build_model(input_dim=1, units=8).to(device)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()

            all_losses.append(ckpt['run_losses'])
            all_models.append(model)
            all_rmse.append(ckpt['rmse'])
            all_mse.append(ckpt['mse'])
            all_nmse.append(ckpt['nmse'])
            all_predictions.append(ckpt['predictions'])
            print(f"Loaded run {run + 1} -- RMSE: {ckpt['rmse']:.6f}, MSE: {ckpt['mse']:.6f}, NMSE: {ckpt['nmse']:.10f} "
                  f"(best val RMSE: {ckpt['best_val_rmse']:.6f})")
            continue

        print(f'\n===== RUN {run + 1}/{N_RUNS} =====')

        np.random.seed(run)
        torch.manual_seed(run)
        model = build_model(input_dim=1, units=8).to(device)

        with torch.no_grad():
            hs = model.hidden_size
            model.cell.bias.data[hs:2 * hs] = 1.0

        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()
        run_losses = []

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        n_samples = X_train_t.size(0)

        best_val_rmse = float('inf')
        best_state = None
        patience_ctr = 0

        for epoch in range(100):
            model.train()
            model.reset_states(1, device)
            epoch_loss = 0.0

            for i in range(n_samples):
                x_i = X_train_t[i:i + 1]
                y_i = y_train_t[i:i + 1]

                optimizer.zero_grad()
                pred = model(x_i)
                loss = criterion(pred.squeeze(-1), y_i)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # FIX: stabilize online updates
                optimizer.step()

                model.u = model.u.detach()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / n_samples
            run_losses.append(avg_loss)

            # FIX: early stopping on the validation split (previously computed, never used)
            _, val_rmse, _, _ = evaluate_on_test(model, val_X, train, train_scaled, raw_values, scaler, device)
            print(f"Epoch {epoch + 1}/100 — train loss: {avg_loss:.6f}  val RMSE: {val_rmse:.6f}")

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"Early stopping at epoch {epoch + 1} (best val RMSE: {best_val_rmse:.6f})")
                    break

        model.load_state_dict(best_state)  # restore best-val checkpoint
        model.reset_states(1, device)      # clear stale hidden state before storing

        all_losses.append(run_losses)
        all_models.append(model)
        print(f'Training complete for run {run + 1} (best val RMSE: {best_val_rmse:.6f})')

        predictions, rmse, mse, nmse = evaluate_on_test(
            model, X_test, train, train_scaled, raw_values, scaler, device
        )

        all_rmse.append(rmse)
        all_mse.append(mse)
        all_nmse.append(nmse)
        all_predictions.append(predictions)

        print(f'Run {run + 1} — RMSE: {rmse:.6f}, MSE: {mse:.6f}, NMSE: {nmse:.10f}')

        # Checkpoint this run to disk immediately, so a crash later never
        # loses this run's work.
        torch.save({
            'model_state_dict': model.state_dict(),
            'run_losses': run_losses,
            'rmse': rmse,
            'mse': mse,
            'nmse': nmse,
            'predictions': predictions,
            'best_val_rmse': best_val_rmse,
        }, run_ckpt_path)

    # Consolidated checkpoint (metrics + all 60 state_dicts), so a fully
    # completed run-set never needs to be retrained even if the individual
    # per-run checkpoint files above are deleted/cleaned up later.
    torch.save({
        'rmse': all_rmse,
        'mse': all_mse,
        'nmse': all_nmse,
        'predictions': all_predictions,
        'losses': all_losses,
        'model_state_dicts': [m.state_dict() for m in all_models],
    }, summary_path)


--- [PyTorch] RUNNING ON cuda ---

Checkpoints will be read/written under: c:\Users\paulp\OneDrive\Desktop\fuzzy_LSTM\with noise\type_1\type_1_baseline_test_60\checkpoints_sp500


===== RUN 1/60 =====
Epoch 1/100 — train loss: 0.059236  val RMSE: 101.567472
Epoch 2/100 — train loss: 0.056278  val RMSE: 100.536430
Epoch 3/100 — train loss: 0.056245  val RMSE: 100.473344
Epoch 4/100 — train loss: 0.056184  val RMSE: 100.445833
Epoch 5/100 — train loss: 0.056133  val RMSE: 100.425057
Epoch 6/100 — train loss: 0.056089  val RMSE: 100.408068
Epoch 7/100 — train loss: 0.056049  val RMSE: 100.393736
Epoch 8/100 — train loss: 0.056013  val RMSE: 100.381411
Epoch 9/100 — train loss: 0.055980  val RMSE: 100.370665
Epoch 10/100 — train loss: 0.055949  val RMSE: 100.361191
Epoch 11/100 — train loss: 0.055920  val RMSE: 100.352760
Epoch 12/100 — train loss: 0.055892  val RMSE: 100.345191
Epoch 13/100 — train loss: 0.055866  val RMSE: 100.338342
Epoch 14/100 — train loss: 0.055841  val RMSE: 100.33

In [13]:
#Noise-robustness sweep (averaged over multiple noise draws, with explicit clean baseline)

N_NOISE_DRAWS = 10  # average over multiple ε(t) realizations per model, per level

# Re-evaluate the clean (0% noise) baseline through evaluate_on_test, so it
# uses the exact same evaluation path as the noisy conditions below.
clean_rmse, clean_mse, clean_nmse = [], [], []
for model in all_models:
    _, rmse, mse, nmse = evaluate_on_test(model, X_test, train, train_scaled, raw_values, scaler, device)
    clean_rmse.append(rmse)
    clean_mse.append(mse)
    clean_nmse.append(nmse)

print(f"[{'no noise':>10}]  "
      f"RMSE: {np.mean(clean_rmse):.6f} ± {np.std(clean_rmse):.6f}  |  "
      f"MSE: {np.mean(clean_mse):.6f} ± {np.std(clean_mse):.6f}  |  "
      f"NMSE: {np.mean(clean_nmse):.10f} ± {np.std(clean_nmse):.10f}")

noise_robustness = {}  # noise_level -> dict of per-model RMSE/MSE/NMSE (averaged over draws)

for noise_level in NOISE_LEVELS:
    level_rmse, level_mse, level_nmse = [], [], []

    for run, model in enumerate(all_models):
        draw_rmse, draw_mse, draw_nmse = [], [], []
        for draw in range(N_NOISE_DRAWS):
            seed = hash((noise_level, run, draw)) % (2 ** 32)
            X_test_noisy = add_gaussian_noise(X_test, noise_level, ref_std, seed=seed)
            _, rmse, mse, nmse = evaluate_on_test(
                model, X_test_noisy, train, train_scaled, raw_values, scaler, device
            )
            draw_rmse.append(rmse)
            draw_mse.append(mse)
            draw_nmse.append(nmse)

        level_rmse.append(np.mean(draw_rmse))  # this model's average over noise draws
        level_mse.append(np.mean(draw_mse))
        level_nmse.append(np.mean(draw_nmse))

    noise_robustness[noise_level] = {'rmse': level_rmse, 'mse': level_mse, 'nmse': level_nmse}
    print(f"[{noise_level * 100:5.1f}% noise]  "
          f"RMSE: {np.mean(level_rmse):.6f} ± {np.std(level_rmse):.6f}  |  "
          f"MSE: {np.mean(level_mse):.6f} ± {np.std(level_mse):.6f}  |  "
          f"NMSE: {np.mean(level_nmse):.10f} ± {np.std(level_nmse):.10f}")

[  no noise]  RMSE: 58.184484 ± 0.455202  |  MSE: 3385.641353 ± 52.683889  |  NMSE: 0.0200775337 ± 0.0003124261
[  0.5% noise]  RMSE: 58.184429 ± 0.455126  |  MSE: 3385.634971 ± 52.674709  |  NMSE: 0.0200774959 ± 0.0003123716
[  5.0% noise]  RMSE: 58.184209 ± 0.453847  |  MSE: 3385.608509 ± 52.524347  |  NMSE: 0.0200773389 ± 0.0003114799
[ 10.0% noise]  RMSE: 58.187502 ± 0.457815  |  MSE: 3385.996514 ± 53.001539  |  NMSE: 0.0200796399 ± 0.0003143098
[ 15.0% noise]  RMSE: 58.186596 ± 0.458424  |  MSE: 3385.892091 ± 53.063715  |  NMSE: 0.0200790206 ± 0.0003146785
